In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Feature Extraction Setup
model = models.alexnet()
num_ftrs = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_ftrs, 7)
model.load_state_dict(torch.load("fine_tuned_alexnet.pth"))

# Expose FC7 by removing the last layer (Linear) and the ReLU/Dropout before it
# In torchvision alexnet, classifier is: 0:Drop, 1:Lin(4096), 2:ReLU, 3:Drop, 4:Lin(4096), 5:ReLU, 6:Lin(7)
# We want the output of layer 4 (FC7) 
model.classifier = nn.Sequential(*list(model.classifier.children())[:5])
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Assume we have a function that loads segments grouped by their original utterance
# For demonstration, let's say `audio_clip_segments` is a tensor of shape (N_segments, 3, 227, 227)
def extract_fc7(segments_tensor):
    with torch.no_grad():
        # Output shape: (N_segments, 4096)
        return model(segments_tensor.to(device)).cpu().numpy()

# 2. Discriminant Temporal Pyramid Matching (DTPM)
def lp_norm_pooling(matrix, p=1.12):
    # Formula: (1/N * sum(|x|^p)) ^ (1/p) [cite: 302]
    N = matrix.shape[0]
    return np.power(np.mean(np.power(np.abs(matrix), p), axis=0), 1/p)

def dtpm(segment_features, L=2):
    N = segment_features.shape[0]
    final_feature = []
    
    # L=0: Whole utterance [cite: 299]
    final_feature.append(lp_norm_pooling(segment_features) * (1 / (2**L))) # Weighting [cite: 311]
    
    # L=1: Halves [cite: 300]
    if N >= 2:
        mid = N // 2
        final_feature.append(lp_norm_pooling(segment_features[:mid]) * (1 / (2**L)))
        final_feature.append(lp_norm_pooling(segment_features[mid:]) * (1 / (2**L)))
        
    # L=2: Quarters
    if L == 2 and N >= 4:
        q_size = N // 4
        for i in range(4):
            start = i * q_size
            end = (i + 1) * q_size if i != 3 else N
            final_feature.append(lp_norm_pooling(segment_features[start:end]) * (1 / (2**(L-1))))
            
    # Concatenate all pooled vectors into a single global representation [cite: 308-309]
    return np.concatenate(final_feature)

# 3. SVM Classification [cite: 413, 426]
# In practice, you would run extract_fc7() and dtpm() on all your training audio clips to create X_train_global
# X_train_global shape will be (Num_Audio_Clips, 28672) if L=2 (7 blocks * 4096)
svm_classifier = SVC(kernel='linear', decision_function_shape='ovo')
# svm_classifier.fit(X_train_global, y_train_utterance_labels)
# predictions = svm_classifier.predict(X_test_global)